# Greenhouse–Geisser A–E analysis — slide & publication figures

Reads the four tables from the A–E repeated-measures analysis and exports **16:9 slides** (PNG 300 dpi + editable SVG) plus **PDFs** for the paper.

```text
data/A_to_E_GG_omnibus.csv
data/A_to_E_GG_posthoc.csv
data/A_to_E_GG_posthoc_significant.csv
data/A_to_E_GG_master_summary.csv
```

| Slide | Content |
|---|---|
| `G1_omnibus` | All 24 AOI × metric tests: partial η², GG + FDR significance, one shared x-axis |
| `G2_profiles_significant` | Condition means A–E for the tests with a significant omnibus effect, with post-hoc brackets |
| `G2_profiles_<AOI>` | Same for all 8 metrics of one AOI (appendix) |
| `G3_C_forest` | How C differs from A, B, D, E: paired d<sub>z</sub> ± bootstrap 95 % CI |
| `G4_pairwise_matrices` | All 10 pairwise d<sub>z</sub> per significant test as colour matrices |

### What changed compared with the previous version
- **Consistent colours:** one fixed, colour-blind-safe colour per condition (Okabe–Ito palette), used on every slide. Previously the forest plot took colours from matplotlib's rotating cycle, so "C vs A" had a different colour in each panel and the legend could not be trusted.
- **d<sub>z</sub> direction is verified from the data** instead of assumed: the sign convention of `dz` is checked against the condition means before any comparison is flipped to "C − other".
- **Shared x-axis** in the omnibus figure, so bubble positions are comparable across AOIs (each panel had its own scale before).
- **Only tested rows** in the forest plot, so there are no empty metric rows.
- **Readable labels:** q-values are no longer printed inside the bubbles, `q = 0.000` becomes `q < .001`, and titles no longer overlap subtitles.
- **Post-hoc brackets** with significance stars on the mean profiles.
- **New pairwise-matrix slide** that uses all 70 post-hoc rows, not only the C comparisons.
- **Error bars** are drawn automatically if the tables contain SD / SE / CI columns per condition.

In [ ]:
# %pip install pandas numpy matplotlib
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

## 1. Configuration

In [ ]:
OMNIBUS_CSV = "data/A_to_E_GG_omnibus.csv"
POSTHOC_CSV = "data/A_to_E_GG_posthoc.csv"
POSTHOC_SIG_CSV = "data/A_to_E_GG_posthoc_significant.csv"
MASTER_CSV = "data/A_to_E_GG_master_summary.csv"

AOIS = ["CTA", "Headline", "Product"]
METRICS = [
    "VAI (%)", "TTFF (s)", "Time spent (s)", "Fixations",
    "Fix. duration (s)", "FFD (s)", "K-coefficient", "Revisits",
]
CONDITIONS = ["A", "B", "C", "D", "E"]
FOCUS = "C"                      # condition the story is about

ALPHA = 0.05

# ---- output ----
OUT_DIR = Path("slides_gg")
SLIDE_SIZE = (13.333, 7.5)       # 16:9
SLIDE_DPI = 300
EXPORT_FORMATS = ("png", "svg", "pdf")   # svg = editable in PowerPoint, pdf = paper

# ---- options ----
BRACKETS = "focus"               # post-hoc brackets on profiles: "focus" (pairs with C), "all", or None
ERROR_BARS = "auto"              # "auto" = use SD/SE/CI columns if the tables have them; None = off
PROFILE_Y_FROM_ZERO = False      # True = y-axis starts at 0 (honest scale, flatter look)

## 2. Load and check the tables

In [ ]:
def load(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"Could not find: {path}")
    df = pd.read_csv(path)
    df.columns = [str(c).replace("**", "").strip() for c in df.columns]
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.replace("**", "", regex=False).str.strip()
    return df


def as_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


omnibus = load(OMNIBUS_CSV)
posthoc = load(POSTHOC_CSV)
posthoc_sig = load(POSTHOC_SIG_CSV)
master = load(MASTER_CSV)

omnibus["significant_GG_FDR"] = as_bool(omnibus["significant_GG_FDR"])
if "significant_posthoc_global_FDR" in posthoc:
    posthoc["significant_posthoc_global_FDR"] = as_bool(posthoc["significant_posthoc_global_FDR"])
else:
    posthoc["significant_posthoc_global_FDR"] = False

for name, frame in [("omnibus", omnibus), ("posthoc", posthoc),
                    ("posthoc_sig", posthoc_sig), ("master", master)]:
    print(f"{name:12s} {len(frame):3d} rows | columns: {list(frame.columns)}")

# ---- optional columns, detected by name ----
def find_col(frame, patterns):
    for pat in patterns:
        for c in frame.columns:
            if re.fullmatch(pat, c, flags=re.I):
                return c
    return None

P_OMNI = find_col(omnibus, [r"p_GG", r"p[-_ ]?GG[-_ ]?corr", r"p_?unc_?GG", r"p_corr"])
EPS_COL = find_col(omnibus, [r"eps(ilon)?(_GG)?", r"GG_?eps(ilon)?", r"eps_?gg"])
Q_POST = find_col(posthoc, [r"q_posthoc_global_FDR", r"q_.*posthoc.*", r"q_.*global.*", r"q_FDR", r"q"])


def spread_cols(frame):
    """Per-condition uncertainty columns: ('ci', lo, hi), ('sd', col) or ('se', col)."""
    for kind, pats in [("se", ["sem_{c}", "se_{c}"]), ("sd", ["sd_{c}", "std_{c}"])]:
        for pat in pats:
            cols = {c: pat.format(c=c) for c in CONDITIONS}
            if all(v in frame.columns for v in cols.values()):
                return kind, cols
    lo = {c: f"ci_low_{c}" for c in CONDITIONS}
    hi = {c: f"ci_high_{c}" for c in CONDITIONS}
    if all(v in frame.columns for v in list(lo.values()) + list(hi.values())):
        return "ci", (lo, hi)
    return None, None

SPREAD_KIND, SPREAD_COLS = (spread_cols(omnibus) if ERROR_BARS == "auto" else (None, None))

print("\nOptional columns found:")
print("  omnibus p (GG):", P_OMNI, "| GG epsilon:", EPS_COL,
      "| post-hoc q:", Q_POST, "| error bars:", SPREAD_KIND or "none")

missing = [(a, m) for a in AOIS for m in METRICS
           if omnibus[(omnibus.AOI == a) & (omnibus.metric == m)].empty]
if missing:
    print("⚠ Missing omnibus rows:", missing)

## 3. Check the sign convention of `dz`

A paired d<sub>z</sub> has the same sign as the difference of the two condition means. Comparing the sign of `dz` with `mean_group1 − mean_group2` tells us whether the table stores `group1 − group2` or `group2 − group1`, so every comparison can be flipped correctly to **C − other**.

In [ ]:
om_idx = omnibus.set_index(["AOI", "metric"])

def mean_of(aoi, metric, cond):
    return float(om_idx.loc[(aoi, metric), f"mean_{cond}"])

diffs = posthoc.apply(lambda r: mean_of(r.AOI, r.metric, r.group1)
                      - mean_of(r.AOI, r.metric, r.group2), axis=1)
usable = (diffs.abs() > 1e-12) & (posthoc["dz"].abs() > 1e-12)
agree = (np.sign(posthoc.loc[usable, "dz"]) == np.sign(diffs[usable])).mean()

if agree >= 0.8:
    DZ_SIGN = +1
    print(f"✓ dz = group1 − group2  (sign agrees with the means in {agree:.0%} of rows)")
elif agree <= 0.2:
    DZ_SIGN = -1
    print(f"✓ dz = group2 − group1  (sign opposite to group1 − group2 in {1 - agree:.0%} of rows)")
else:
    DZ_SIGN = +1
    print(f"⚠ Sign convention unclear ({agree:.0%} agreement) — assuming group1 − group2. "
          "Check how dz was computed.")

ph_idx = {(r.AOI, r.metric, r.group1, r.group2): r for r in posthoc.itertuples(index=False)}


def pair_effect(aoi, metric, a, b):
    """dz, CI and significance for (a − b), whatever order the table stores."""
    r = ph_idx.get((aoi, metric, a, b))
    flip = 1
    if r is None:
        r = ph_idx.get((aoi, metric, b, a))
        flip = -1
    if r is None:
        return None
    s = DZ_SIGN * flip
    lo, hi = sorted([s * r.dz_ci_low, s * r.dz_ci_high])
    q = getattr(r, Q_POST) if Q_POST else np.nan
    return dict(dz=s * r.dz, lo=lo, hi=hi,
                sig=bool(r.significant_posthoc_global_FDR), q=q)


tested = [(a, m) for a in AOIS for m in METRICS
          if any(k[0] == a and k[1] == m for k in ph_idx)]
print(f"{len(tested)} AOI × metric tests have post-hoc comparisons")

## 4. Style and helpers

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "svg.fonttype": "none",      # SVG text stays editable
    "pdf.fonttype": 42,          # TrueType in PDF: editable in Illustrator, accepted by journals
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#9CA3AF",
})

TEXT, MUTED, GRID = "#1F2937", "#6B7280", "#E5E7EB"
SIG, NS = "#2563EB", "#A3A9B3"
RAW = "#F59E0B"

# Okabe–Ito (colour-blind safe); the focus condition gets the strongest colour
OKABE = ["#E69F00", "#009E73", "#0072B2", "#D55E00", "#CC79A7"]
COND_COLORS = dict(zip(CONDITIONS, OKABE))
if FOCUS in COND_COLORS:
    COND_COLORS[FOCUS], COND_COLORS[CONDITIONS[2]] = "#0072B2", COND_COLORS[FOCUS]


def fmt_p(x, lead=""):
    if pd.isna(x):
        return ""
    return f"{lead}< .001" if x < 0.001 else f"{lead}= " + f"{x:.3f}".lstrip("0")


def stars(q):
    if pd.isna(q):
        return "*"
    return "***" if q < .001 else "**" if q < .01 else "*" if q < ALPHA else "n.s."


def short(m):
    return re.sub(r"\s*\(.*?\)", "", m)


def add_title(fig, title, subtitle=None):
    fig.text(0.035, 0.955, title, fontsize=28, fontweight="bold", color=TEXT, va="top")
    if subtitle:
        fig.text(0.035, 0.885, subtitle, fontsize=17, color=MUTED, va="top")


def add_footnote(fig, text):
    fig.text(0.035, 0.018, text, fontsize=11, color=MUTED, va="bottom")


def save(fig, name):
    OUT_DIR.mkdir(exist_ok=True)
    for ext in EXPORT_FORMATS:
        fig.savefig(OUT_DIR / f"{name}.{ext}", dpi=SLIDE_DPI, facecolor="white")
    plt.show()
    print("Saved:", ", ".join(f"{name}.{e}" for e in EXPORT_FORMATS))


def omni_row(aoi, metric):
    return om_idx.loc[(aoi, metric)]


def omni_status(row):
    if bool(row["significant_GG_FDR"]):
        return "sig"
    if P_OMNI and row[P_OMNI] < ALPHA:
        return "raw"
    return "ns"

## 5. Slide G1 — Omnibus overview

In [ ]:
n_sig = int(omnibus["significant_GG_FDR"].sum())
by_aoi = omnibus[omnibus.significant_GG_FDR].groupby("AOI").size().reindex(AOIS, fill_value=0)
subtitle = (f"{n_sig} of {len(omnibus)} tests show a condition effect after GG + FDR correction  "
            f"({' · '.join(f'{a} {n}' for a, n in by_aoi.items())})")

XMAX = max(0.2, np.ceil(omnibus["partial_eta_sq"].max() * 1.15 / 0.05) * 0.05)
colors = {"sig": SIG, "raw": RAW, "ns": NS}

fig, axes = plt.subplots(1, 3, figsize=SLIDE_SIZE, sharex=True, sharey=True)
fig.subplots_adjust(left=0.155, right=0.975, top=0.78, bottom=0.24, wspace=0.07)

for i, (ax, aoi) in enumerate(zip(axes, AOIS)):
    for lo, hi, col in [(0.01, 0.06, "#F9FAFB"), (0.06, 0.14, "#F3F4F6"), (0.14, XMAX, "#EBEDF0")]:
        ax.axvspan(lo, hi, color=col, zorder=0, lw=0)
        ax.axvline(lo, ls=(0, (3, 3)), lw=1, color="#B6BBC2", zorder=1)
    for y, metric in enumerate(METRICS):
        if (aoi, metric) not in om_idx.index:
            continue
        r = omni_row(aoi, metric)
        eta, st = float(r["partial_eta_sq"]), omni_status(r)
        col = colors[st]
        ax.hlines(y, 0, eta, color=col, lw=3.2 if st != "ns" else 2.2, zorder=2)
        ax.scatter(eta, y, s=230 if st == "sig" else 150,
                   facecolor=col if st != "ns" else "white", edgecolor=col,
                   lw=2.2, zorder=4, clip_on=False)
        label = f"{eta:.2f}" + (f"   q {fmt_p(r['q_GG_FDR'])}" if st == "sig" else "")
        inside = eta > XMAX * 0.5          # long bars: put the label on the left of the point
        ax.annotate(label, (eta, y), xytext=(-14 if inside else 11, 0),
                    textcoords="offset points", va="center",
                    ha="right" if inside else "left", fontsize=12,
                    fontweight="bold" if st == "sig" else "normal",
                    color=TEXT if st == "sig" else MUTED, zorder=5,
                    bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5) if inside else None)
    ax.set_yticks(range(len(METRICS)))
    ax.set_yticklabels(METRICS, color=TEXT)
    ax.set_ylim(len(METRICS) - 0.5, -0.5)
    ax.set_xlim(0, XMAX)
    ax.xaxis.set_major_locator(plt.MaxNLocator(5, prune="upper"))  # no clash at panel joins
    ax.grid(axis="x", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0, pad=14, labelleft=(i == 0))
    ax.tick_params(axis="x", colors=MUTED)
    ax.set_title(aoi, loc="left", fontsize=20, fontweight="bold", color=TEXT, pad=12)

fig.supxlabel(r"Partial $\eta^2$ (effect of condition A–E)", fontsize=16, color=TEXT, y=0.13)
add_title(fig, "Where do conditions A–E change viewing behaviour?", subtitle)

handles = [
    Line2D([0], [0], marker="o", ls="", ms=11, mfc="white", mec=NS, mew=2, label="n.s."),
    Line2D([0], [0], marker="o", ls="", ms=12, mfc=SIG, mec=SIG, label="GG + FDR q < .05"),
]
if P_OMNI:
    handles.insert(1, Line2D([0], [0], marker="o", ls="", ms=11, mfc=RAW, mec=RAW,
                             label="GG p < .05, not after FDR"))
fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.028, 0.045),
           ncol=len(handles), frameon=False, columnspacing=1.8, handletextpad=0.4)

note = "Shaded zones: partial η² small ≥ .01 · medium ≥ .06 · large ≥ .14   |   repeated-measures ANOVA, Greenhouse–Geisser corrected"
if EPS_COL:
    note += f"   |   GG ε range {omnibus[EPS_COL].min():.2f}–{omnibus[EPS_COL].max():.2f}"
add_footnote(fig, note)
save(fig, "G1_omnibus")

## 6. Slides G2 — Mean profiles with post-hoc brackets

In [ ]:
def spread(row, cond):
    """(lower, upper) error-bar lengths for one condition, or None."""
    if SPREAD_KIND in ("sd", "se"):
        v = float(row[SPREAD_COLS[cond]])
        return v, v
    if SPREAD_KIND == "ci":
        lo, hi = SPREAD_COLS
        m = float(row[f"mean_{cond}"])
        return m - float(row[lo[cond]]), float(row[hi[cond]]) - m
    return None


def draw_profile(ax, aoi, metric, title=None, small=False):
    row = omni_row(aoi, metric)
    x = np.arange(len(CONDITIONS))
    means = np.array([float(row[f"mean_{c}"]) for c in CONDITIONS])
    errs = [spread(row, c) for c in CONDITIONS]

    ax.plot(x, means, color="#D1D5DB", lw=1.6, zorder=1)
    for xi, c, m, e in zip(x, CONDITIONS, means, errs):
        if e is not None:
            ax.errorbar(xi, m, yerr=[[e[0]], [e[1]]], color=COND_COLORS[c],
                        lw=1.6, capsize=3, zorder=2)
        ax.scatter(xi, m, s=(150 if c == FOCUS else 90) * (0.8 if small else 1),
                   color=COND_COLORS[c], edgecolor="white", lw=1.2, zorder=3)

    lows = [m - (e[0] if e else 0) for m, e in zip(means, errs)]
    highs = [m + (e[1] if e else 0) for m, e in zip(means, errs)]
    ymin, ymax = (0 if PROFILE_Y_FROM_ZERO else min(lows)), max(highs)
    span = (ymax - ymin) or abs(ymax) or 1

    # post-hoc brackets
    top = ymax
    if BRACKETS:
        pairs = [(a, b) for i, a in enumerate(CONDITIONS) for b in CONDITIONS[i + 1:]
                 if BRACKETS == "all" or FOCUS in (a, b)]
        sig_pairs = []
        for a, b in pairs:
            eff = pair_effect(aoi, metric, a, b)
            if eff and eff["sig"]:
                sig_pairs.append((a, b, eff["q"]))
        sig_pairs.sort(key=lambda t: abs(CONDITIONS.index(t[0]) - CONDITIONS.index(t[1])))
        step = span * 0.13
        for k, (a, b, q) in enumerate(sig_pairs):
            x1, x2 = CONDITIONS.index(a), CONDITIONS.index(b)
            yb = ymax + step * (k + 0.7)
            ax.plot([x1, x1, x2, x2], [yb - step * 0.2, yb, yb, yb - step * 0.2],
                    color=TEXT, lw=1.1, zorder=4)
            ax.text((x1 + x2) / 2, yb, stars(q), ha="center", va="bottom",
                    fontsize=11 if small else 12, color=TEXT)
            top = yb + step * 0.55
    ax.set_ylim(ymin - span * 0.08, max(top, ymax + span * 0.08))

    ax.set_xticks(x)
    ax.set_xticklabels(CONDITIONS, fontweight="bold")
    for lbl, c in zip(ax.get_xticklabels(), CONDITIONS):
        lbl.set_color(COND_COLORS[c])
    ax.set_xlim(-0.5, len(CONDITIONS) - 0.5)
    ax.grid(axis="y", color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", labelsize=11 if small else 12, colors=MUTED)
    ax.tick_params(axis="x", length=0, labelsize=12 if small else 13)

    sig = bool(row["significant_GG_FDR"])
    ax.set_title(title or metric, loc="left", fontsize=14 if small else 15,
                 fontweight="bold", color=TEXT if sig else MUTED, pad=22)
    ax.text(0, 1.035, f"q {fmt_p(row['q_GG_FDR'])}" + ("  ✓" if sig else ""),
            transform=ax.transAxes, fontsize=11.5, va="bottom",
            color=SIG if sig else MUTED, fontweight="bold" if sig else "normal")
    if not sig:
        ax.set_facecolor("#FAFAFA")


def profile_legend(fig, y=0.045):
    handles = [Line2D([0], [0], marker="o", ls="", ms=11 if c == FOCUS else 9,
                      mfc=COND_COLORS[c], mec="white", label=f"Condition {c}")
               for c in CONDITIONS]
    extra = "   |   brackets: post-hoc FDR  * q < .05  ** q < .01  *** q < .001" if BRACKETS else ""
    fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.028, y),
               ncol=len(handles), frameon=False, columnspacing=1.4, handletextpad=0.2)
    return extra


spread_note = {"sd": "error bars = ±1 SD", "se": "error bars = ±1 SE",
               "ci": "error bars = 95 % CI", None: "points = condition means"}[SPREAD_KIND]

# ---- significant tests (chunks of 8) ----
sig_tests = [(a, m) for a in AOIS for m in METRICS
             if (a, m) in om_idx.index and bool(omni_row(a, m)["significant_GG_FDR"])]
for part, start in enumerate(range(0, len(sig_tests), 8), start=1):
    chunk = sig_tests[start:start + 8]
    ncol = min(4, len(chunk)); nrow = int(np.ceil(len(chunk) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=SLIDE_SIZE, squeeze=False)
    fig.subplots_adjust(left=0.06, right=0.975, top=0.73 if nrow > 1 else 0.7,
                        bottom=0.16, wspace=0.28, hspace=0.62)
    for ax, (a, m) in zip(axes.flat, chunk):
        draw_profile(ax, a, m, title=f"{a} · {short(m)}", small=True)
    for ax in list(axes.flat)[len(chunk):]:
        ax.axis("off")
    extra = profile_legend(fig)
    suffix = f" ({part})" if len(sig_tests) > 8 else ""
    add_title(fig, f"What the significant effects look like{suffix}",
              f"Condition means for the {len(sig_tests)} tests with a GG + FDR omnibus effect; "
              f"condition {FOCUS} highlighted")
    add_footnote(fig, spread_note + extra)
    save(fig, "G2_profiles_significant" + (f"_{part}" if len(sig_tests) > 8 else ""))

# ---- appendix: one slide per AOI ----
for aoi in AOIS:
    fig, axes = plt.subplots(2, 4, figsize=SLIDE_SIZE)
    fig.subplots_adjust(left=0.06, right=0.975, top=0.73, bottom=0.16, wspace=0.3, hspace=0.62)
    for ax, m in zip(axes.flat, METRICS):
        if (aoi, m) in om_idx.index:
            draw_profile(ax, aoi, m, small=True)
        else:
            ax.axis("off")
    extra = profile_legend(fig)
    n = int(omnibus[(omnibus.AOI == aoi)]["significant_GG_FDR"].sum())
    add_title(fig, f"{aoi}: mean gaze profiles across A–E",
              f"{n} of {len(METRICS)} metrics show a GG + FDR condition effect (dark titles); "
              "grey panels are n.s.")
    add_footnote(fig, spread_note + extra)
    save(fig, f"G2_profiles_{aoi}")

## 7. Slide G3 — How {FOCUS} differs from the other conditions

In [ ]:
others = [c for c in CONDITIONS if c != FOCUS]
rows = []
for a, m in tested:
    for o in others:
        eff = pair_effect(a, m, FOCUS, o)
        if eff:
            rows.append(dict(AOI=a, metric=m, other=o, **eff))
forest = pd.DataFrame(rows)

if forest.empty:
    print("No post-hoc comparisons with", FOCUS, "— slide skipped.")
else:
    n_groups = len(tested)
    offs = np.linspace(-0.3, 0.3, len(others))
    L = max(1.0, np.ceil(max(forest["hi"].abs().max(), forest["lo"].abs().max()) * 1.1 * 2) / 2)

    fig, ax = plt.subplots(figsize=SLIDE_SIZE)
    fig.subplots_adjust(left=0.235, right=0.975, top=0.78, bottom=0.2)

    # alternating row bands keep the 4 comparisons of one test visually together
    for g in range(n_groups):
        if g % 2 == 0:
            ax.axhspan(g - 0.5, g + 0.5, color="#F3F4F6", zorder=0, lw=0)
    ax.grid(axis="x", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.axvline(0, color=TEXT, lw=1.2, zorder=1)

    for g, (a, m) in enumerate(tested):
        for off, o in zip(offs, others):
            r = forest[(forest.AOI == a) & (forest.metric == m) & (forest.other == o)]
            if r.empty:
                continue
            r = r.iloc[0]
            y = g + off
            col = COND_COLORS[o]
            ax.hlines(y, r.lo, r.hi, color=col, lw=2.4 if r.sig else 1.6,
                      alpha=1 if r.sig else 0.55, zorder=2)
            ax.scatter(r.dz, y, s=90 if r.sig else 55, zorder=3,
                       facecolor=col if r.sig else "white", edgecolor=col, lw=1.8)

    ax.set_yticks(range(n_groups))
    ax.set_yticklabels([f"{a} · {m}" for a, m in tested], color=TEXT, fontsize=14)
    ax.set_ylim(n_groups - 0.5, -0.5)
    ax.set_xlim(-L, L)
    ax.tick_params(axis="y", length=0, pad=10)
    ax.tick_params(axis="x", colors=MUTED)
    ax.set_xlabel(rf"Paired effect size $d_z$  ({FOCUS} − other)", color=TEXT, labelpad=8)
    ax.text(-L * 0.98, -0.62, f"◀ {FOCUS} lower", fontsize=12, color=MUTED, va="bottom")
    ax.text(L * 0.98, -0.62, f"{FOCUS} higher ▶", fontsize=12, color=MUTED, va="bottom", ha="right")

    n_s = int(forest.sig.sum())
    add_title(fig, f"How condition {FOCUS} differs from the others",
              f"{n_s} of {len(forest)} {FOCUS}-comparisons are significant after post-hoc FDR "
              f"(filled markers)")
    handles = [Line2D([0], [0], marker="o", color=COND_COLORS[o], lw=2.4, ms=9,
                      label=f"{FOCUS} vs {o}") for o in others]
    handles += [Line2D([0], [0], marker="o", ls="", ms=9, mfc="white", mec=MUTED, mew=1.8,
                       label="n.s. (hollow)")]
    fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.028, 0.045),
               ncol=len(handles), frameon=False, columnspacing=1.6, handletextpad=0.5)
    add_footnote(fig, "Point = paired dz · line = bootstrap 95 % CI · |dz| ≈ .2 small, .5 medium, .8 large   |   "
                      "rows = tests with a significant omnibus effect")
    save(fig, "G3_C_forest")

## 8. Slide G4 — All pairwise differences as matrices

In [ ]:
if not tested:
    print("No post-hoc data — slide skipped.")
else:
    k = len(CONDITIONS)
    mats = {}
    for a, m in tested:
        M = np.full((k, k), np.nan); S = np.zeros((k, k), bool)
        for i in range(k):
            for j in range(i):
                eff = pair_effect(a, m, CONDITIONS[i], CONDITIONS[j])
                if eff:
                    M[i, j], S[i, j] = eff["dz"], eff["sig"]
        mats[(a, m)] = (M, S)

    L = np.nanmax([np.nanmax(np.abs(M)) for M, _ in mats.values()])
    L = max(0.5, np.ceil(L * 2) / 2)
    cmap = LinearSegmentedColormap.from_list("div", ["#2166AC", "#F7F7F7", "#B2182B"])
    cmap.set_bad("white")
    norm = TwoSlopeNorm(vmin=-L, vcenter=0, vmax=L)

    per_slide = 8
    for part, start in enumerate(range(0, len(tested), per_slide), start=1):
        chunk = tested[start:start + per_slide]
        fig, axes = plt.subplots(2, 4, figsize=SLIDE_SIZE, squeeze=False)
        fig.subplots_adjust(left=0.05, right=0.89, top=0.74, bottom=0.12, wspace=0.18, hspace=0.42)
        for ax, (a, m) in zip(axes.flat, chunk):
            M, S = mats[(a, m)]
            im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, norm=norm)
            for i in range(k):
                for j in range(i):
                    if np.isnan(M[i, j]):
                        continue
                    v = M[i, j]
                    txt = ax.text(j, i, f"{v + 0.0:.1f}".replace("-0.0", "0.0") + ("*" if S[i, j] else ""),
                                  ha="center", va="center", fontsize=10.5,
                                  fontweight="bold" if S[i, j] else "normal",
                                  color="white" if abs(v) > 0.6 * L else TEXT)
                    if S[i, j]:
                        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                                   ec=TEXT, lw=1.6))
            ax.set_xticks(range(k)); ax.set_yticks(range(k))
            ax.set_xticklabels(CONDITIONS, fontsize=11, fontweight="bold")
            ax.set_yticklabels(CONDITIONS, fontsize=11, fontweight="bold")
            for lbl, c in zip(ax.get_xticklabels(), CONDITIONS):
                lbl.set_color(COND_COLORS[c])
            for lbl, c in zip(ax.get_yticklabels(), CONDITIONS):
                lbl.set_color(COND_COLORS[c])
            ax.set_xlim(-0.5, k - 1.5); ax.set_ylim(k - 0.5, 0.5)  # drop empty row/col
            ax.tick_params(length=0)
            for s in ax.spines.values():
                s.set_visible(False)
            ax.set_title(f"{a} · {short(m)}", fontsize=13.5, fontweight="bold",
                         color=TEXT, loc="left", pad=8)
        for ax in list(axes.flat)[len(chunk):]:
            ax.axis("off")

        cax = fig.add_axes([0.915, 0.2, 0.017, 0.5])
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax)
        cb.set_label(r"$d_z$ (row − column)", fontsize=13, color=TEXT)
        cb.ax.tick_params(labelsize=11, colors=MUTED)
        cb.outline.set_visible(False)

        suffix = f" ({part})" if len(tested) > per_slide else ""
        add_title(fig, f"All pairwise differences{suffix}",
                  "Paired dz for every pair of conditions — red: row condition higher, blue: lower")
        add_footnote(fig, "* and outlined = significant after post-hoc FDR   |   "
                          "only tests with a significant GG + FDR omnibus effect")
        save(fig, "G4_pairwise_matrices" + (f"_{part}" if len(tested) > per_slide else ""))

## Outputs

Everything is written to `slides_gg/` as PNG (slides), SVG (editable slides) and PDF (paper). Change `EXPORT_FORMATS` to drop any of them.